# T15 — Thử nghiệm PhoBERT có/không tách từ

Notebook chạy trên Google Colab GPU. Hai biến thể chỉ khác tiền xử lý: `normalize(text)` và `word_tokenize(normalize(text), format='text')`. Cả hai dùng cùng dữ liệu, seed và siêu tham số T09; test chỉ được đọc sau khi hai checkpoint đã khóa bằng dev. Không thay đổi `normalize()` và không đưa `underthesea` vào backend.

In [ ]:
import importlib.metadata as metadata, subprocess, sys
required = {'transformers': '5.18.0', 'scikit-learn': '1.7.2', 'underthesea': '9.5.0'}
for package, version in required.items():
    try:
        installed = metadata.version(package)
    except metadata.PackageNotFoundError:
        installed = None
    if installed != version:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', f'{package}=={version}'])
print({name: metadata.version(name) for name in required})


In [ ]:
from pathlib import Path
import hashlib, json, os, random, shutil, sys, time

from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive')
root_candidates = [Path(os.environ['FIREFLY_PROJECT_ROOT'])] if os.environ.get('FIREFLY_PROJECT_ROOT') else []
cwd = Path.cwd().resolve()
root_candidates.extend([cwd, *cwd.parents, Path('/content/firefly-cinema')])
PROJECT_ROOT = next((p.resolve() for p in root_candidates if (p / 'ml/src/eval_utils.py').is_file()), None)
code_archive = DRIVE_ROOT / 'thesis/code/firefly-cinema-T15-code.zip'
if PROJECT_ROOT is None and code_archive.is_file():
    shutil.unpack_archive(code_archive, '/content/firefly-cinema')
    PROJECT_ROOT = Path('/content/firefly-cinema').resolve()
assert PROJECT_ROOT is not None, 'Không tìm thấy repo; đặt FIREFLY_PROJECT_ROOT hoặc upload code archive T15'
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / 'ml/src'))
print('PROJECT_ROOT =', PROJECT_ROOT)


In [ ]:
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import f1_score, precision_score, recall_score
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup
from underthesea import word_tokenize
from eval_utils import normalize

os.environ['MODEL_DIR'] = '__t15_normalize_only_missing__'
os.environ['SKLEARN_MODELS'] = ''
from backend.app.classifier import normalize as runtime_normalize

SEED = 42
MAX_LENGTH = 256
BATCH_SIZE = 8
LEARNING_RATE = 2e-5
EPOCHS = 5
PATIENCE = 2
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
MODEL_NAME = 'vinai/phobert-base-v2'
DEVICE = torch.device('cuda')
assert torch.cuda.is_available(), 'T15 phải chạy trên Colab GPU'

def seed_everything():
    os.environ['PYTHONHASHSEED'] = str(SEED)
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
    torch.use_deterministic_algorithms(True, warn_only=True)

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def first_dir(required, *candidates):
    found = next((Path(p) for p in candidates if all((Path(p) / name).is_file() for name in required)), None)
    assert found is not None, f'Thiếu {required}'
    return found

PHONE_DIR = first_dir(('train.csv', 'dev.csv', 'test.csv'), PROJECT_ROOT / 'data/uit-visfd/processed', DRIVE_ROOT / 'thesis/data/processed')
MOVIE_DIR = first_dir(('movie_test.csv',), PROJECT_ROOT / 'data/phim', DRIVE_ROOT / 'thesis/data/movie', DRIVE_ROOT / 'thesis/data/phim')
RESULT_PATH = PROJECT_ROOT / 'ml/results/word_segment.csv'
DRIVE_RESULT_DIR = DRIVE_ROOT / 'thesis_results/T15'
DRIVE_MODEL_DIR = DRIVE_ROOT / 'thesis_results/T15_word_segment'
DRIVE_RESULT_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_MODEL_DIR.mkdir(parents=True, exist_ok=True)
seed_everything()
print('GPU =', torch.cuda.get_device_name(0))


## Gate tiền xử lý và dữ liệu

Dùng nguyên văn `normalize()` hiện hành qua `ml/src/eval_utils.py`, đối chiếu runtime trên 20 câu. Train/dev là split chính thức của UIT-ViSFD; chưa đọc phone test hoặc movie test.

In [ ]:
train_raw = pd.read_csv(PHONE_DIR / 'train.csv', encoding='utf-8-sig')
dev_raw = pd.read_csv(PHONE_DIR / 'dev.csv', encoding='utf-8-sig')
for frame, rows, name in ((train_raw, 7786, 'train'), (dev_raw, 1112, 'dev')):
    assert len(frame) == rows and {'text_raw', 'has_negative'} <= set(frame.columns), name
    assert frame[['text_raw', 'has_negative']].notna().all().all()
    assert set(frame['has_negative'].astype(int)) == {0, 1}
parity = pd.DataFrame({'text': train_raw['text_raw'].head(20).astype(str)})
parity['notebook'] = parity['text'].map(normalize)
parity['runtime'] = parity['text'].map(runtime_normalize)
parity['equal'] = parity['notebook'] == parity['runtime']
display(parity)
assert len(parity) == 20 and parity['equal'].all()

def no_segment(text):
    return normalize(str(text))

def with_segment(text):
    return word_tokenize(normalize(str(text)), format='text')

PREPROCESSORS = {'without_segmentation': no_segment, 'with_segmentation': with_segment}
preview = parity[['text']].head(10).copy()
preview['without_segmentation'] = preview['text'].map(no_segment)
preview['with_segmentation'] = preview['text'].map(with_segment)
display(preview)
print('NORMALIZE_PARITY_OK=20/20; TEST_NOT_LOADED=True')


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
TRIAL_ROOT = Path('/content/t15_trials')
TRIAL_ROOT.mkdir(parents=True, exist_ok=True)

def prepared(frame, preprocess):
    result = pd.DataFrame({'text': frame['text_raw'].map(preprocess), 'label': frame['has_negative'].astype(int)})
    assert not result['text'].eq('').any()
    return result

class TextDataset(Dataset):
    def __init__(self, frame):
        self.texts = frame['text'].astype(str).tolist(); self.labels = frame['label'].astype(int).tolist()
    def __len__(self): return len(self.labels)
    def __getitem__(self, index): return self.texts[index], self.labels[index]

def collate(batch):
    texts, labels = zip(*batch)
    encoded = tokenizer(list(texts), padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors='pt')
    encoded['labels'] = torch.tensor(labels, dtype=torch.long)
    return encoded

def scores(labels, predictions):
    return {
        'precision_neg': precision_score(labels, predictions, pos_label=1, zero_division=0),
        'recall_neg': recall_score(labels, predictions, pos_label=1, zero_division=0),
        'f1_neg': f1_score(labels, predictions, pos_label=1, zero_division=0),
        'macro_f1': f1_score(labels, predictions, average='macro', zero_division=0),
    }

def predict(model, frame):
    loader = DataLoader(TextDataset(frame), batch_size=16, shuffle=False, collate_fn=collate)
    output = []; model.eval()
    with torch.inference_mode():
        for batch in loader:
            batch.pop('labels')
            logits = model(**{k: v.to(DEVICE) for k, v in batch.items()}).logits
            output.extend(torch.argmax(logits, dim=-1).cpu().tolist())
    return np.asarray(output, dtype=int)

def train_variant(name, preprocess):
    seed_everything()
    train = prepared(train_raw, preprocess); dev = prepared(dev_raw, preprocess)
    generator = torch.Generator().manual_seed(SEED)
    loader = DataLoader(TextDataset(train), batch_size=BATCH_SIZE, shuffle=True, generator=generator, collate_fn=collate)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(DEVICE)
    model.config.id2label = {0: 'binh_thuong', 1: 'tieu_cuc'}
    model.config.label2id = {'binh_thuong': 0, 'tieu_cuc': 1}
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    total_steps = EPOCHS * len(loader)
    scheduler = get_linear_schedule_with_warmup(optimizer, int(total_steps * WARMUP_RATIO), total_steps)
    scaler = torch.amp.GradScaler('cuda', enabled=True)
    destination = TRIAL_ROOT / name
    best_score = (-1.0, -1.0); best_epoch = 0; stale = 0; history = []
    for epoch in range(1, EPOCHS + 1):
        model.train(); total_loss = 0.0
        for batch in loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda', dtype=torch.float16):
                loss = model(**batch).loss
            scaler.scale(loss).backward(); scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            old_scale = scaler.get_scale(); scaler.step(optimizer); scaler.update()
            if scaler.get_scale() >= old_scale:
                scheduler.step()
            total_loss += float(loss.detach().cpu())
        metric = scores(dev['label'], predict(model, dev))
        history.append({'epoch': epoch, 'train_loss': total_loss / len(loader), **metric})
        candidate = (metric['macro_f1'], metric['f1_neg'])
        print(name, history[-1])
        if candidate > best_score:
            best_score = candidate; best_epoch = epoch; stale = 0
            model.save_pretrained(destination); tokenizer.save_pretrained(destination)
        else:
            stale += 1
            if stale >= PATIENCE: break
    del model, optimizer, scheduler, scaler
    torch.cuda.empty_cache()
    return {'variant': name, 'best_epoch': best_epoch, 'dev_macro_f1': best_score[0], 'dev_f1_neg': best_score[1], 'history': history}


## Train và khóa selection

Hai lượt train dùng cùng thứ tự dữ liệu và siêu tham số. Chỉ preprocessing khác nhau; checkpoint tốt nhất được chọn bằng phone dev, không dùng test.

In [ ]:
runs = [train_variant(name, preprocess) for name, preprocess in PREPROCESSORS.items()]
manifest = {
    'seed': SEED, 'model': MODEL_NAME, 'max_length': MAX_LENGTH, 'batch_size': BATCH_SIZE,
    'learning_rate': LEARNING_RATE, 'max_epochs': EPOCHS, 'patience': PATIENCE,
    'weight_decay': WEIGHT_DECAY, 'warmup_ratio': WARMUP_RATIO,
    'train_sha256': sha256(PHONE_DIR / 'train.csv'), 'dev_sha256': sha256(PHONE_DIR / 'dev.csv'),
    'selection_rule': 'max phone dev macro_f1, then f1_neg; test not loaded',
    'runs': runs, 'test_loaded': False,
}
manifest_path = TRIAL_ROOT / 'selection_manifest.json'
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
display(pd.DataFrame([{k: v for k, v in run.items() if k != 'history'} for run in runs]))
print('SELECTION_LOCKED=True; TEST_NOT_LOADED=True')


## Final test — chỉ chạy sau `SELECTION_LOCKED`

Đọc đúng phone test 2.224 câu và movie test 149 câu. Đồng thời đo chi phí tiền xử lý trên toàn bộ 2.373 câu bằng ba lượt, báo median ms/câu.

In [ ]:
locked = json.loads(manifest_path.read_text(encoding='utf-8'))
assert locked['test_loaded'] is False
phone_test_raw = pd.read_csv(PHONE_DIR / 'test.csv', encoding='utf-8-sig')
movie_test_raw = pd.read_csv(MOVIE_DIR / 'movie_test.csv', encoding='utf-8-sig')
assert len(phone_test_raw) == 2224 and phone_test_raw['has_negative'].value_counts().sort_index().to_dict() == {0: 981, 1: 1243}
assert len(movie_test_raw) == 149 and movie_test_raw['label'].value_counts().sort_index().to_dict() == {0: 48, 1: 101}
raw_texts = phone_test_raw['text_raw'].astype(str).tolist() + movie_test_raw['text'].astype(str).tolist()

def median_ms(preprocess):
    samples = []
    for _ in range(3):
        start = time.perf_counter()
        [preprocess(text) for text in raw_texts]
        samples.append((time.perf_counter() - start) * 1000 / len(raw_texts))
    return float(np.median(samples))

preprocess_ms = {name: median_ms(fn) for name, fn in PREPROCESSORS.items()}
additional_ms = preprocess_ms['with_segmentation'] - preprocess_ms['without_segmentation']
rows = []
for run in runs:
    name = run['variant']; preprocess = PREPROCESSORS[name]
    model = AutoModelForSequenceClassification.from_pretrained(TRIAL_ROOT / name, local_files_only=True).to(DEVICE)
    tests = {
        'phone': pd.DataFrame({'text': phone_test_raw['text_raw'].map(preprocess), 'label': phone_test_raw['has_negative'].astype(int)}),
        'movie': pd.DataFrame({'text': movie_test_raw['text'].map(preprocess), 'label': movie_test_raw['label'].astype(int)}),
    }
    for test_set, frame in tests.items():
        metric = scores(frame['label'], predict(model, frame))
        rows.append({
            'variant': name, 'test_set': test_set, **{k: round(v, 6) for k, v in metric.items()},
            'preprocess_ms_per_sentence': round(preprocess_ms[name], 6),
            'additional_segmentation_ms_per_sentence': round(additional_ms if name == 'with_segmentation' else 0.0, 6),
            'seed': SEED, 'max_length': MAX_LENGTH, 'batch_size': BATCH_SIZE,
            'learning_rate': LEARNING_RATE, 'max_epochs': EPOCHS, 'best_epoch': run['best_epoch'],
        })
    destination = DRIVE_MODEL_DIR / name
    shutil.copytree(TRIAL_ROOT / name, destination, dirs_exist_ok=True)
    del model; torch.cuda.empty_cache()
results = pd.DataFrame(rows)
expected_pairs = {(v, t) for v in PREPROCESSORS for t in ('phone', 'movie')}
assert set(zip(results['variant'], results['test_set'])) == expected_pairs
display(results)
print('ADDITIONAL_SEGMENTATION_MS_PER_SENTENCE=', round(additional_ms, 6))


In [ ]:
previous_path = DRIVE_RESULT_DIR / 'word_segment_previous.csv'
if previous_path.is_file():
    previous = pd.read_csv(previous_path).set_index(['variant', 'test_set'])
    current = results.set_index(['variant', 'test_set'])
    deltas = [abs(float(current.loc[index, metric]) - float(previous.loc[index, metric])) for index in current.index for metric in ('f1_neg', 'macro_f1')]
    assert max(deltas) <= 0.001, f'Clean-run lệch quá 0,001: {max(deltas)}'
    print('CLEAN_RUN_REPRODUCIBILITY_MAX_DELTA=', max(deltas))
else:
    print('CLEAN_RUN_1_SAVED; chạy lại từ runtime mới để kiểm sai lệch <= 0,001')
results.to_csv(RESULT_PATH, index=False, encoding='utf-8', lineterminator='\n')
results.to_csv(DRIVE_RESULT_DIR / 'word_segment.csv', index=False, encoding='utf-8', lineterminator='\n')
results.to_csv(previous_path, index=False, encoding='utf-8', lineterminator='\n')
locked['test_loaded'] = True
locked['test_hashes'] = {'phone': sha256(PHONE_DIR / 'test.csv'), 'movie': sha256(MOVIE_DIR / 'movie_test.csv')}
locked['word_segment_csv_sha256'] = sha256(RESULT_PATH)
(DRIVE_RESULT_DIR / 'selection_manifest.json').write_text(json.dumps(locked, ensure_ascii=False, indent=2), encoding='utf-8')

lookup = results.set_index(['variant', 'test_set'])
phone_delta = float(lookup.loc[('with_segmentation', 'phone'), 'f1_neg'] - lookup.loc[('without_segmentation', 'phone'), 'f1_neg'])
movie_delta = float(lookup.loc[('with_segmentation', 'movie'), 'f1_neg'] - lookup.loc[('without_segmentation', 'movie'), 'f1_neg'])
recommendation = ('Đề xuất mở task riêng để cân nhắc tích hợp web sau PO review; T15 không đổi backend.' if phone_delta > 0 and movie_delta > 0 else 'Không nên đưa bản tách từ vào web ở thời điểm này vì F1 không tăng trên cả hai miền.')
table = results[['variant', 'test_set', 'f1_neg', 'macro_f1']].to_markdown(index=False)
report = f'''# Codex Implementation Report — T15

- Ngày: 2026-10-15
- Nhánh: `task/T15-tach-tu`
- Đổi schema: Không
- Phạm vi: chỉ notebook Colab và artifact ML; không sửa `normalize()`, backend hoặc frontend.

## Bảng F1 có/không tách từ

{table}

- Delta F1 tiêu cực (có - không), phone: `{phone_delta:.6f}`.
- Delta F1 tiêu cực (có - không), movie: `{movie_delta:.6f}`.
- Thời gian tách từ tăng thêm: `{additional_ms:.6f} ms/câu` (median của 3 lượt trên 2.373 câu).

## Kết luận

{recommendation}

## Gate

- Seed/siêu tham số: 42; max length 256; batch 8; LR 2e-5; weight decay 0,01; warmup 10%; tối đa 5 epoch; patience 2.
- Split: phone train/dev/test chính thức; movie test cố định 149 câu; test chỉ đọc sau `SELECTION_LOCKED`.
- Normalize parity: 20/20.
- Tái lập: xem output `CLEAN_RUN_REPRODUCIBILITY_MAX_DELTA`; yêu cầu <= 0,001 sau clean-run thứ hai.
- Drive: `MyDrive/thesis_results/T15/` và `MyDrive/thesis_results/T15_word_segment/`.
- Không tự review, không tự PASS; chờ Claude review độc lập sau khi đủ hai clean-run.
'''
report_path = PROJECT_ROOT / 'reports/2026-10-15/codex_implementation_T15_2026-10-15.md'
report_path.parent.mkdir(parents=True, exist_ok=True)
report_path.write_text(report, encoding='utf-8')
card_path = PROJECT_ROOT / 'ml/model_card_A.md'
card = card_path.read_text(encoding='utf-8')
marker = '\n## T15 — Thử nghiệm tách từ (15/10/2026)\n'
if marker in card:
    card = card.split(marker, 1)[0].rstrip() + '\n'
card += marker + f'''
{table}

Chi phí tách từ tăng thêm `{additional_ms:.6f} ms/câu`. Delta F1 tiêu cực (có - không): phone `{phone_delta:.6f}`, movie `{movie_delta:.6f}`. {recommendation}
'''
card_path.write_text(card, encoding='utf-8')
print('RESULT=', RESULT_PATH)
print('REPORT=', report_path)
print('MODEL_CARD=', card_path)
